
# Gigapath 기반 WSI 양성/악성 분류 (2023 병리 데이터)

- **대상**: 2023년 전처리 결과(`mammary_adenoma_vs_adenocarcinoma_only(2023).parquet`)를 사용해 mammary adenoma vs. mammary adenocarcinoma 이진 분류.
- **모델**: Hugging Face `prov-gigapath/prov-gigapath` 타일 인코더 + 단순 MIL 헤드(Attention) 예시.
- **스토리지 전략**: 로컬 여유 200GB, 외장 스토리지(≥1.1TB)에 모든 중간 산출물(패치, 임베딩, 체크포인트)을 저장하도록 경로를 분리.
- **실행 안내**: 각 셀을 순서대로 실행. GPU/드라이버, OpenSlide, pixman 버전은 미리 설치되어 있다고 가정.



## 0. 필수 패키지 설치
- OpenSlide/pixman은 시스템 단에서 설치되어 있어야 합니다.
- GigaPath 레포를 editable 설치하여 데모 유틸을 재사용합니다. (HF 토큰 필요)


In [ ]:

# 가상환경에 맞춰 필요 시 수정
%pip install --upgrade pip
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install 'timm>=1.0.8' pandas pyarrow scikit-learn openslide-python matplotlib seaborn
%pip install huggingface_hub einops
%pip install git+https://github.com/prov-gigapath/prov-gigapath.git



## 1. 경로/환경 설정
- `EXTERNAL_ROOT`를 외장하드 마운트 지점으로 설정합니다.
- 슬라이드/라벨 parquet/출력 경로를 모두 외장 스토리지로 지정해 로컬 디스크 사용을 최소화합니다.
- HF 토큰을 환경 변수 `HF_TOKEN`에 설정해야 모델 다운로드가 가능합니다.


In [ ]:

import os
from pathlib import Path
import pandas as pd

# 외장하드 마운트 경로 (사용 환경에 맞게 수정)
EXTERNAL_ROOT = Path('/mnt/external')  # 예: '/mnt/ssd_ext' 등
SLIDE_ROOT = EXTERNAL_ROOT / 'slides'   # 폴더 구조: SLIDE_ROOT / S23-XXXXX / *.svs
PARQUET_PATH = Path('PoC/v1/mammary_adenoma_vs_adenocarcinoma_only(2023).parquet')
CACHE_ROOT = EXTERNAL_ROOT / 'gigapath_cache'
TILE_ROOT = CACHE_ROOT / 'tiles'
EMBED_ROOT = CACHE_ROOT / 'embeddings'
CKPT_ROOT = CACHE_ROOT / 'checkpoints'

for path in [CACHE_ROOT, TILE_ROOT, EMBED_ROOT, CKPT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

# HF 토큰 설정 (필요 시 수동 입력)
if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = input('Enter your Hugging Face read token: ').strip()



## 2. 2023년 라벨 데이터 로드 및 필터링
- parquet은 `INSP_RQST_NO`, `FILE_NAME`, `DIAGNOSIS` 등을 포함한다고 가정합니다.
- mammary adenoma / mammary adenocarcinoma만 이진 라벨로 정리합니다.


In [ ]:

df = pd.read_parquet(PARQUET_PATH)

POSITIVE_LABELS = {'mammary adenocarcinoma'}
NEGATIVE_LABELS = {'mammary adenoma'}

filtered = df[df['DIAGNOSIS'].str.lower().isin({*POSITIVE_LABELS, *NEGATIVE_LABELS})].copy()
filtered['label'] = filtered['DIAGNOSIS'].str.lower().apply(lambda x: 1 if x in POSITIVE_LABELS else 0)

print('전체 샘플 수:', len(filtered))
print(filtered[['INSP_RQST_NO', 'FILE_NAME', 'DIAGNOSIS', 'label']].head())



## 3. 슬라이드 존재 여부 확인
- 사용자가 제공한 디렉터리 구조(`INSP_RQST_NO/FILE_NAME.svs`)를 기준으로 실제 파일 경로를 구성합니다.
- 누락된 파일을 early check.


In [ ]:

slide_paths = []
missing = []

for _, row in filtered.iterrows():
    slide_dir = SLIDE_ROOT / str(row['INSP_RQST_NO'])
    slide_path = slide_dir / f"{row['FILE_NAME']}.svs"
    if slide_path.exists():
        slide_paths.append({'slide_id': row['INSP_RQST_NO'], 'path': slide_path, 'label': int(row['label'])})
    else:
        missing.append(slide_path)

print(f"존재 확인: {len(slide_paths)}개 / 누락: {len(missing)}개")
if missing:
    print('예시 누락 경로:', missing[:3])



## 4. 타일링 유틸리티 (메모리·디스크 안전 버전)
- 조직 마스크(Otsu threshold)로 배경을 제거하고 20×/256px 기준으로 타일링.
- 타일 PNG는 외장 스토리지(TILE_ROOT)에 저장.
- 대용량 처리를 위해 generator + 저장 후 필요 시 삭제/재사용.


In [ ]:

import numpy as np
import openslide
from skimage import color, filters
from PIL import Image

PATCH_SIZE = 256
MIN_TISSUE_RATIO = 0.6
LEVEL = 0  # 20x 레벨 가정; 필요 시 slide.level_dimensions 확인 후 조정

def tissue_mask(np_rgb: np.ndarray) -> np.ndarray:
    hsv = color.rgb2hsv(np_rgb)
    saturation = hsv[:, :, 1]
    thresh = filters.threshold_otsu(saturation)
    return saturation > thresh

def iter_tiles(slide: openslide.OpenSlide, stride: int = PATCH_SIZE):
    width, height = slide.level_dimensions[LEVEL]
    for y in range(0, height, stride):
        for x in range(0, width, stride):
            region = slide.read_region((x, y), LEVEL, (PATCH_SIZE, PATCH_SIZE)).convert('RGB')
            arr = np.array(region)
            mask = tissue_mask(arr)
            if mask.mean() < MIN_TISSUE_RATIO:
                continue
            yield x, y, region

def export_tiles(slide_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    slide = openslide.OpenSlide(str(slide_path))
    meta = []
    for tile_idx, (x, y, tile_img) in enumerate(iter_tiles(slide)):
        tile_name = f"tile_{tile_idx:06d}_x{x}_y{y}.png"
        tile_path = out_dir / tile_name
        tile_img.save(tile_path, format='PNG')
        meta.append({'tile_path': str(tile_path), 'x': x, 'y': y, 'level': LEVEL})
    slide.close()
    return meta



## 5. GigaPath 타일 인코더 로드
- HF Hub에서 모델을 다운로드합니다 (`HF_TOKEN` 필요).
- `gigapath` 패키지의 `TileEncoder`를 사용해 임베딩을 계산합니다.


In [ ]:

import torch
from huggingface_hub import hf_hub_download
from gigapath.models import TileEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

ckpt_path = hf_hub_download(
    repo_id='prov-gigapath/prov-gigapath',
    filename='prov-gigapath_tileencoder.pth',
    token=os.environ.get('HF_TOKEN')
)

encoder = TileEncoder.from_pretrained(ckpt_path)
encoder.eval().to(DEVICE)



## 6. 타일 임베딩 추출 및 캐싱
- 배치 단위로 타일을 로드하여 FP16 추론 후 parquet/h5로 저장.
- 스토리지 절약을 위해 float16로 저장.


In [ ]:

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import pyarrow as pa
import pyarrow.parquet as pq

transform = transforms.Compose([
    transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class TileDataset(Dataset):
    def __init__(self, tile_meta):
        self.tile_meta = tile_meta
    def __len__(self):
        return len(self.tile_meta)
    def __getitem__(self, idx):
        entry = self.tile_meta[idx]
        img = Image.open(entry['tile_path']).convert('RGB')
        return transform(img), entry['x'], entry['y']

@torch.inference_mode()
def embed_slide(slide_rec):
    slide_id = slide_rec['slide_id']
    slide_path = slide_rec['path']
    out_dir = TILE_ROOT / str(slide_id)
    tile_meta = export_tiles(slide_path, out_dir)

    ds = TileDataset(tile_meta)
    loader = DataLoader(ds, batch_size=64, num_workers=4, pin_memory=True)

    coords, feats = [], []
    for imgs, xs, ys in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE, dtype=torch.float16):
            emb = encoder(imgs)
        feats.append(emb.cpu().to(torch.float16))
        coords.extend(zip(xs.tolist(), ys.tolist()))

    feat_tensor = torch.cat(feats, dim=0)
    table = pa.Table.from_pydict({
        'x': [c[0] for c in coords],
        'y': [c[1] for c in coords],
        'feature': feat_tensor.numpy().tolist(),
    })

    out_path = EMBED_ROOT / f"{slide_id}.parquet"
    pq.write_table(table, out_path, compression='zstd')
    return out_path



## 7. 간단한 MIL 헤드(Attention) 학습 예시
- 모든 임베딩을 메모리에 올리기 어렵다면 epoch마다 parquet → numpy 로딩 후 배치 학습.
- 여기서는 개념 증명용 소형 Attention MIL 헤드를 정의합니다.


In [ ]:

import math
import torch.nn as nn

class AttentionMIL(nn.Module):
    def __init__(self, in_dim, hidden_dim=256):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.classifier = nn.Linear(in_dim, 1)
    def forward(self, feats):
        attn_score = self.attn(feats)  # (N,1)
        weight = torch.softmax(attn_score, dim=0)
        slide_feat = (weight * feats).sum(dim=0, keepdim=True)
        logit = self.classifier(slide_feat)
        return logit.squeeze(0), weight.squeeze(1)



## 8. 학습 루프 스켈레톤
- 실제 실행 시 train/val split을 parquet에서 나눠 사용합니다.
- 데이터가 크므로 `DataLoader` 대신 슬라이드 단위 lazy 로딩으로 구현합니다.


In [ ]:

import random
from sklearn.model_selection import train_test_split
import torch.optim as optim

train_recs, val_recs = train_test_split(slide_paths, test_size=0.2, random_state=42, stratify=[s['label'] for s in slide_paths])

model = AttentionMIL(in_dim=encoder.output_dim).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 5

@torch.inference_mode()
def load_embed(path: Path):
    table = pq.read_table(path)
    feats = torch.tensor(np.stack(table['feature'].to_numpy()), dtype=torch.float16)
    coords = np.stack([table['x'].to_numpy(), table['y'].to_numpy()], axis=1)
    return feats, coords

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_recs)
    total_loss = 0
    for rec in train_recs:
        embed_path = EMBED_ROOT / f"{rec['slide_id']}.parquet"
        if not embed_path.exists():
            embed_path = embed_slide(rec)
        feats, _ = load_embed(embed_path)
        feats = feats.to(DEVICE)

        logit, _ = model(feats)
        label = torch.tensor([rec['label']], device=DEVICE, dtype=torch.float32)
        loss = criterion(logit, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch}: train_loss={total_loss/len(train_recs):.4f}")

    model.eval()
    correct = 0
    with torch.no_grad():
        for rec in val_recs:
            embed_path = EMBED_ROOT / f"{rec['slide_id']}.parquet"
            if not embed_path.exists():
                embed_path = embed_slide(rec)
            feats, _ = load_embed(embed_path)
            feats = feats.to(DEVICE)
            logit, _ = model(feats)
            pred = (torch.sigmoid(logit) > 0.5).int().item()
            correct += int(pred == rec['label'])
    acc = correct / len(val_recs)
    print(f"Val acc: {acc:.3f}")

torch.save(model.state_dict(), CKPT_ROOT / 'attention_mil.pt')



## 9. 슬라이드 단위 추론 + 패치 중요도 heatmap 초안
- attention weight를 이용해 패치 좌표별 중요도를 얻습니다.
- 추후 썸네일과 overlay 가능하도록 좌표/score를 parquet로 저장.


In [ ]:

import matplotlib.pyplot as plt

@torch.inference_mode()
def predict_slide(rec):
    embed_path = EMBED_ROOT / f"{rec['slide_id']}.parquet"
    if not embed_path.exists():
        embed_path = embed_slide(rec)
    feats, coords = load_embed(embed_path)
    feats = feats.to(DEVICE)
    logit, attn = model(feats)
    prob = torch.sigmoid(logit).item()
    return prob, coords, attn.cpu().numpy()

example = train_recs[0]
prob, coords, attn = predict_slide(example)
print(f"Slide {example['slide_id']} prob(malignant)={prob:.3f}")

plt.figure(figsize=(6, 5))
plt.scatter(coords[:,0], coords[:,1], c=attn, s=10, cmap='hot')
plt.gca().invert_yaxis()
plt.colorbar(label='Attention')
plt.title('Patch importance (higher = more malignant)')
plt.show()



## 10. 후처리/청소
- 타일 PNG를 더 이상 사용하지 않는다면 용량 절약을 위해 삭제하세요.
- parquet 임베딩은 재사용을 위해 남겨둡니다.


In [ ]:

import shutil

REMOVE_TILES = False
if REMOVE_TILES:
    shutil.rmtree(TILE_ROOT, ignore_errors=True)
